# 正则化技术：Early Stopping 与 Label Smoothing

本 notebook 讲解两个治过拟合的常用技术：**Early Stopping（早停）** 和 **Label Smoothing（标签平滑）**。两者都在你工程的 `vit/training.py` 正则化版里启用了。

## 一句话区分

- **Early Stopping**：训到一定程度就停，防止"学过头"
- **Label Smoothing**：标签从"非黑即白"变"留点余地"，防止"过度自信"

一个管训练**时间维度**（训多久），一个管**目标维度**（目标多硬），是两个正交方向的正则。

## 目录

1. Early Stopping：原理 + 你工程里的实现
2. Label Smoothing：原理 + 数值示例
3. 两者怎么配合其他正则手段


## 1. Early Stopping 原理

### 核心思想

训练不是越久越好。模型在训练集上 loss 一直降，但在验证集上 loss 会**先降后升**（U 形）。val_loss 开始反弹的那个点，就是过拟合的起点；早停就是在这附近停下。

### 为什么有效

- 早停时模型还没"记住"训练集噪声 → 泛化更好
- 省训练时间、省算力
- 不需要改模型结构，零成本

### 关键参数

- **patience（耐心）**：val_loss 连续多少个 epoch 不降才停。太小容易误停（val_loss 抖动），太大没意义
- **min_delta（最小改善）**：val_loss 下降多少算"实质改善"。防止抖动假象

### 直觉

想象考试刷题：刷越多训练分越高，但真考分会在某个点后下降（死记硬背替代了理解）。早停 = 在真考分最高那个点停手。


## 你工程里的实现（vit/training.py）

新版 `fit()` 里关键逻辑（简化版）：

```python
# val_loss 至少下降 min_delta 才算实质改善
validation_loss_improved = validation_loss < best_validation_loss - min_delta

if validation_loss_improved:
    best_validation_loss = validation_loss
    epochs_without_improvement = 0
    # 同时保存 best checkpoint（按最低 val_loss）
    torch.save(...)
else:
    epochs_without_improvement += 1
    if epochs_without_improvement >= patience:
        break  # 早停
```

### 你的配置（config.py）

- `EARLY_STOPPING_PATIENCE = 20`：val_loss 连续 20 epoch 不降就停
- `EARLY_STOPPING_MIN_DELTA = 1e-4`：下降不到 0.0001 不算改善

### patience=20 是大是小

- 偏宽松：给了模型 20 epoch "挣扎"的机会，不容易因抖动误停
- 对 150 epoch 的总训练量，20 epoch ≈ 13%，合理
- 看到 val_loss 抖动剧烈想更激进 → 调到 10；想更保守 → 调到 30

### best checkpoint 也变了

旧版按 `val_acc 最高` 选 best；新版按 `val_loss 最低` 选。配合早停，best 就是过拟合起点那个权重，不是后期 acc 抖动高点的那个。


## 2. Label Smoothing 原理

### 核心思想

普通标签是 one-hot：`[0, 0, 1, 0, ..., 0]`（猫那张图标签里 cat=1，其他=0，非黑即白）。
Label Smoothing 把它"软化"：`[0.011, 0.011, 0.91, 0.011, ..., 0.011]`（cat=0.91，其他各类各 0.01）。

公式（K 类、平滑系数 α）：

```
soft_label = (1 - α) * one_hot + α / K
```

- α=0：退化成普通 one-hot
- α=0.1：真实类 0.91，其他各类 0.01（10 类时）

### 为什么有效

- **防过度自信**：旧版模型把 cat 判成 truck 时给 99.8% 概率，就是"过度自信"的典型。Label smoothing 让目标分布本身就不那么尖，模型学到的概率也更平
- **改善校准**：预测概率更接近真实准确率（高置信度预测更可能真对）
- **抗噪**：训练标签可能有错标，soft label 让模型对单标签不那么笃定

### 直觉

教小孩分类猫狗：硬标签像说"这绝对是猫，没别的可能"；软标签像说"这很可能是一只猫，但我也留 1% 给它可能是别的"。后者让小孩学得更稳健，不会因为见过一只长得像狗的猫就崩溃。


## PyTorch 里的实现（vit/training.py）

一行就够：

```python
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
```

### 它在干什么

PyTorch 内部把 CrossEntropyLoss 的目标从 one-hot 换成 soft label：

- 原来：`loss = -log(softmax(logits)[true_class])`，只盯真实类
- 现在：`loss = -( (1-α) * log(p[true]) + α/K * Σ log(p[i]) )`，所有类都有一点目标

等价于在真实类上加 (1-α) 权重、在所有类上各加 α/K 权重。

### α 数值选择

- **0**：退化成普通 CE
- **0.05 ~ 0.1**：常用，ViT 论文用 0.1
- **0.15 ~ 0.2**：训练标签噪声多时可以加大
- **>0.2**：通常太软，欠拟合风险

你工程用 0.1，是标准值。

### 你工程的对应症状

旧版有 cat→truck 99.8%、cat→ship 96.9% 这种"高置信度错判"。Label smoothing 直接治这个——目标分布变软后，模型学到的预测分布也变软，cat→truck 这种极端概率会被拉低。


## 3. 两者怎么配合其他正则手段

正则化版 TinyViT 同时用了 5 个正则手段，每个打在过拟合的不同侧面：

| 手段 | 治什么 | 在哪个文件 |
|------|--------|-----------|
| Dropout 0.2 | 神经元共适应（互相依赖） | config.py |
| Label Smoothing 0.1 | 过度自信（概率太尖） | training.py |
| Early Stopping | 学过头（val_loss 反弹） | training.py |
| RandAugment + RandomErasing | 记死训练样本 | data.py |
| Weight Decay 0.05 | 权重长得太大 | AdamW |

### 互不替代、各有侧重

- Dropout 治"特征共适应"（神经元互相依赖）
- Label Smoothing 治"概率过度自信"
- Early Stopping 治"训太久"
- 数据增强 治"死记样本"
- Weight Decay 治"权重过大"

五个一起上 = 多角度压制过拟合。这正是 CHANGES.txt 说"先跑这版观察，如果还有 gap 再加 Mixup"的原因——已经够全了，Mixup 是后备。

### 你该怎么观察效果

重训后看三件事：

1. **val_loss 曲线**是否还在反弹（反弹幅度变小 = 正则起效）
2. **train/val gap** 是否从 12pt 缩小
3. **误分类样本**里"高置信度错判"（cat→truck 99.8%）是否变少 = label smoothing 起效


## 小结

### Early Stopping
- val_loss 不降就停，零成本治"学过头"
- 关键参数 patience（等多久）、min_delta（多算改善）
- 你工程配置：patience=20、min_delta=1e-4

### Label Smoothing
- 标签从 one-hot 软化，治"过度自信"
- 一行 `nn.CrossEntropyLoss(label_smoothing=0.1)` 启用
- 你工程配置：α=0.1

### 两者配合

- 早停管"训多久"（时间维度）
- 平滑管"目标多硬"（目标维度）
- 加上 dropout/增强/weight decay = 五个正交方向一起压过拟合

### 一句话

**早停 = 在过拟合起点附近停手；label smoothing = 让目标标签别那么绝对。一个管时间，一个管目标，配合使用 = 从两个正交方向压过拟合。**
